# Battery — cross-cell transfer: train on B0006, forecast B0005

## The question

Every battery number we have was trained and tested on the same cell. That leaves one thing
unmeasured: whether the network learned **degradation dynamics** or just memorised one cell's
trajectory. Training on B0006 and forecasting B0005 answers it directly, and it is the strongest
evidence we can offer that the model generalises.

The bar is not our 0.00850. The bar is the **flat-line baseline on B0005's forecast region**. A model
that has never seen B0005 and still beats persistence on it has learned something real about how
lithium-ion cells fade. That is the claim worth making, and it is a different claim from being
accurate.

## Protocol — unchanged from the B0005 work

Everything is imported from the existing study rather than rebuilt: window 10, multi-scale CNN
(kernels 3/5/7) concatenated with a bidirectional LSTM, delta targets accumulated recursively,
300 epochs, patience 30, batch 16, 5 seeds. SOH is capacity divided by that cell's own first
capacity, so both cells arrive in the same units without any fitted scaler — which is what makes a
clean transfer possible at all.

Forecast origin is index `int(0.8*N)-1` on the target cell, the same 80% split point used throughout.

## Three tiers, because "transfer" means three different things

The tiers differ by **how much of the target cell is allowed**, and each answers its own question.

**Tier 0 — no B0005 data whatsoever.** The network trains on B0006 and forecasts B0005 cold, seeded
only by the ten SOH values immediately before the origin (which the protocol already grants every
model, including the incumbent).

- `A1` zero-shot, plain

**Tier 1 — B0005's pre-origin cycles used for calibration only; the network never trains on B0005.**
This is the realistic deployment case: a new cell is installed, you observe it for a while, but you
do not retrain. B0006 fades faster than B0005, so an uncorrected transfer will over-predict fade —
these two conditions test whether that offset is the whole problem.

- `A2` physics: fade curve re-anchored on B0005's last 20 pre-origin cycles, residual dynamics from B0006
- `A3` calibrated: predicted deltas rescaled by the ratio of the two cells' pre-origin fade rates

**Tier 2 — B0005's pre-origin cycles used for training.** Here transfer has to prove it adds
something over simply using the data you have.

- `B1` within-cell only — **this is the incumbent and the control**
- `B2` B0006 pre-training, then fine-tuned on B0005 pre-origin
- `B3` pooled: B0006 plus B0005 pre-origin trained together

`B1` is what makes `B2` and `B3` interpretable. Without it, "fine-tuning worked" cannot be
distinguished from "the B0005 data worked".

## Reverse direction

The same ladder, abbreviated, training on B0005 and forecasting B0006. If transfer works one way and
not the other, that is about the cells, not the method — B0006 is the noisier, faster-fading cell and
is genuinely harder. Reporting only the favourable direction would be a choice we would have to
defend later.

In [1]:
import os, gc, time, json, math, random, warnings
os.environ['PYTHONHASHSEED']='42'
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from scipy.io import loadmat
from scipy.optimize import curve_fit
from scipy.stats import ttest_rel
from sklearn.metrics import mean_squared_error
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import tensorflow as tf
for _g in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(_g,True)
    except RuntimeError as _e: print('memory growth:',_e)
from tensorflow.keras import backend as K
from tensorflow.keras.layers import (Input, Conv1D, MaxPooling1D, Dropout, Dense, LSTM,
                                     Bidirectional, Concatenate, GlobalAveragePooling1D)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
def preflight():
    if not tf.config.list_physical_devices('GPU'): print('CPU mode'); return False
    try:
        x=np.random.rand(8,10,1).astype('float32'); y=np.random.rand(8,1).astype('float32')
        i=Input(shape=(10,1)); o=Dense(1)(Bidirectional(LSTM(4))(i))
        m=Model(i,o); m.compile('adam','mse'); m.fit(x,y,epochs=1,verbose=0); K.clear_session()
        print('GPU preflight passed'); return True
    except Exception as e:
        print('GPU preflight FAILED (%s) -> CPU'%type(e).__name__); K.clear_session(); return False
USE_GPU=preflight(); DEVICE='/GPU:0' if USE_GPU else '/CPU:0'
try: tf.config.experimental.enable_op_determinism()
except Exception as e: print('determinism:',e)
print('TF',tf.__version__,'| device',DEVICE)

WINDOW=10; SEED=42; EPOCHS, PATIENCE, BATCH = 300, 30, 16
FT_EPOCHS, FT_PATIENCE, FT_LR = 100, 20, 1e-4
N_SEEDS=5; FADE_L=20

GPU preflight passed
TF 2.10.0 | device /GPU:0


## Load both cells and quantify how different they are

If the two cells were interchangeable, transfer would be trivial and uninteresting. Measuring the
shift first tells us what any transfer method has to overcome.

In [2]:
def load_cell(name):
    mat=loadmat(name+'.mat'); cycles=mat[name][0,0]['cycle'][0]; caps=[]
    for i in range(len(cycles)):
        r=cycles[i]
        if r['type'][0]!='discharge': continue
        caps.append(float(r['data'][0][0]['Capacity'][0][0]))
    CAP=np.asarray(caps,np.float64); C0=CAP[0]
    S=(CAP/C0).astype(np.float32); N=len(S); START=int(0.8*N)-1
    return dict(name=name,CAP=CAP,C0=C0,S=S,N=N,START=START,H=N-START)

CELLS={n:load_cell(n) for n in ('B0005','B0006')}
def f_lin(k,a,b): return a*k+b
def fit_fade(S,origin,L=FADE_L):
    a=max(0,origin-L); k=np.arange(a,origin,dtype=float)
    par,_=curve_fit(f_lin,k,S[a:origin],p0=[-1e-3,1.0],maxfev=20000)
    return par

print('%-7s %5s %8s %9s %9s %11s %11s'%('cell','N','C0 (Ah)','S_end','origin','fade/cycle','delta sd'))
for n,c in CELLS.items():
    d=np.diff(c['S'][:c['START']])
    c['fade']=fit_fade(c['S'],c['START'])[0]
    c['FLOOR']=float(np.sqrt(mean_squared_error(c['S'][c['START']:],
                     np.full(c['H'],c['S'][c['START']-1]))))
    print('%-7s %5d %8.4f %9.4f %9d %11.6f %11.6f'
          %(n,c['N'],c['C0'],c['S'][-1],c['START'],c['fade'],d.std()))
r=CELLS['B0005']['fade']/CELLS['B0006']['fade']
print('\npre-origin fade rate ratio  B0005 / B0006 = %.3f'%r)
print('   -> a B0006-trained model applied cold to B0005 should over-predict fade by ~%.0f%%'%(100*(1/r-1)))
print('\nflat-line baselines on the forecast region (the bar to beat):')
for n,c in CELLS.items(): print('   %-7s %.5f   over %d cycles'%(n,c['FLOOR'],c['H']))

cell        N  C0 (Ah)     S_end    origin  fade/cycle    delta sd
B0005     168   1.8565    0.7138       133   -0.001833    0.007293
B0006     168   2.0353    0.5825       133   -0.001697    0.012339

pre-origin fade rate ratio  B0005 / B0006 = 1.081
   -> a B0006-trained model applied cold to B0005 should over-predict fade by ~-7%

flat-line baselines on the forecast region (the bar to beat):
   B0005   0.02927   over 35 cycles
   B0006   0.04608   over 35 cycles


In [3]:
def build(window=WINDOW, n_feat=1, n_out=1, filters=32, ms_kernels=(3,5,7), lstm_units=32,
          n_cnn=1, n_dense=1, dense_units=64, dropout=0.1, lr=1e-3, seed=SEED):
    """Round 1's architecture, unchanged — imported, not rebuilt."""
    kw=dict(kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seed))
    inp=Input(shape=(window,n_feat)); c=inp
    for _ in range(n_cnn):
        c=Concatenate()([Conv1D(filters,k,padding='same',activation='relu',**kw)(c) for k in ms_kernels])
        c=Dropout(dropout,seed=seed)(c)
        if int(c.shape[1])>2: c=MaxPooling1D(2,padding='same')(c)
    c=GlobalAveragePooling1D()(c)
    r=Bidirectional(LSTM(lstm_units,**kw))(inp)
    h=Concatenate()([c,r])
    for _ in range(n_dense):
        h=Dense(dense_units,activation='relu',**kw)(h); h=Dropout(dropout,seed=seed)(h)
    m=Model(inp,Dense(n_out,activation='linear',**kw)(h))
    m.compile(optimizer=tf.keras.optimizers.Adam(lr),loss='mse'); return m

def windows(series, idxs):
    X=np.asarray([series[e-WINDOW:e] for e in idxs],np.float32).reshape(-1,WINDOW,1)
    y=np.asarray([series[e]-series[e-1] for e in idxs],np.float32).reshape(-1,1)
    return X,y

def fit_model(Xf,yf,Xv,yv,seed,model=None,epochs=EPOCHS,patience=PATIENCE,lr=None):
    # NO clear_session here: several models must stay alive within one seed, because B2
    # fine-tunes the very network A1 was rolled out from. The session is cleared once per
    # seed instead, at the end of the loop.
    np.random.seed(seed); tf.random.set_seed(seed)
    with tf.device(DEVICE):
        m=build(seed=seed) if model is None else model
        if lr is not None: K.set_value(m.optimizer.lr, lr)
        m.fit(Xf,yf,validation_data=(Xv,yv),epochs=epochs,batch_size=BATCH,verbose=0,shuffle=True,
              callbacks=[EarlyStopping('val_loss',patience=patience,restore_best_weights=True),
                         ReduceLROnPlateau('val_loss',factor=0.5,patience=10,min_lr=1e-7)])
    return m

def train_on(series, idxs, seed, model=None, **kw):
    Xa,ya=windows(series,idxs); c=int(0.85*len(Xa))
    return fit_model(Xa[:c],ya[:c],Xa[c:],ya[c:],seed,model=model,**kw)

def rollout(m, seed_series, origin, H, scale=1.0, base=None, lo=0.0, hi=1.2):
    """Recursive forecast. `base` adds a physics curve back; `scale` rescales predicted deltas."""
    win=list(seed_series[origin-WINDOW:origin]); out=[]; prev=float(seed_series[origin-1])
    for _ in range(H):
        x=np.asarray(win[-WINDOW:],np.float32).reshape(1,WINDOW,1)
        d=float(m(x,training=False).numpy().ravel()[0])*scale
        prev=float(np.clip(prev+d,lo,hi)) if base is None else float(prev+d)
        out.append(prev); win.append(prev)
    out=np.asarray(out,np.float32)
    return np.clip(out if base is None else base+out, 0.0, 1.2)

def rmse(tgt,pred): return float(np.sqrt(mean_squared_error(tgt,pred)))
print('helpers ready | trainable params:',build().count_params())

helpers ready | trainable params: 19649


## The ladder

`SRC` is the cell the network learns from; `TGT` is the cell it is scored on. Everything about the
scoring — origin, horizon, metric, flat-line baseline — belongs to `TGT` and is identical to the
incumbent study, so the numbers drop straight into the existing tables.

In [4]:
def run_direction(src_name, tgt_name, seeds=N_SEEDS, verbose=True):
    SRC, TGT = CELLS[src_name], CELLS[tgt_name]
    S_s, S_t = SRC['S'], TGT['S']
    o, H = TGT['START'], TGT['H']
    tgt_true = S_t[o:]
    IDX_SRC_ALL = [e for e in range(WINDOW, SRC['N'])]          # the whole source cell is fair game
    IDX_TGT_PRE = [e for e in range(WINDOW, o)]                 # target cell, pre-origin only

    # physics pieces, both anchored on their OWN cell's pre-origin cycles
    par_t = fit_fade(S_t, o); CURVE_T = np.clip(f_lin(np.arange(float(TGT['N'])),*par_t),0,1.2).astype(np.float32)
    par_s = fit_fade(S_s, SRC['START']); CURVE_S = np.clip(f_lin(np.arange(float(SRC['N'])),*par_s),0,1.2).astype(np.float32)
    RESID_S = (S_s-CURVE_S).astype(np.float32); RESID_T = (S_t-CURVE_T).astype(np.float32)
    curve_only = rmse(tgt_true, CURVE_T[o:])
    scale_cal = float(par_t[0]/par_s[0])                        # fade-rate ratio, from pre-origin data only

    rows={k:[] for k in ['A1','A2','A3','B1','B2','B3']}; preds={}
    for s in range(seeds):
        sd=SEED+s
        # order matters: A1/A3 must roll out m_src BEFORE B2 fine-tunes it in place
        m_src   = train_on(S_s, IDX_SRC_ALL, sd)                       # plain, source cell
        p=rollout(m_src,S_t,o,H);                       rows['A1'].append(rmse(tgt_true,p)); preds.setdefault('A1',p)
        p=rollout(m_src,S_t,o,H,scale=scale_cal);       rows['A3'].append(rmse(tgt_true,p)); preds.setdefault('A3',p)
        m_res   = train_on(RESID_S, IDX_SRC_ALL, sd)                   # residual dynamics, source cell
        p=rollout(m_res,RESID_T,o,H,base=CURVE_T[o:]);  rows['A2'].append(rmse(tgt_true,p)); preds.setdefault('A2',p)
        m_within= train_on(S_t, IDX_TGT_PRE, sd)                       # the incumbent / control
        p=rollout(m_within,S_t,o,H);                    rows['B1'].append(rmse(tgt_true,p)); preds.setdefault('B1',p)
        m_ft    = train_on(S_t, IDX_TGT_PRE, sd, model=m_src,
                           epochs=FT_EPOCHS, patience=FT_PATIENCE, lr=FT_LR)
        p=rollout(m_ft,S_t,o,H);                        rows['B2'].append(rmse(tgt_true,p)); preds.setdefault('B2',p)
        Xa,ya=windows(S_s,IDX_SRC_ALL); Xb,yb=windows(S_t,IDX_TGT_PRE)
        Xp,yp=np.concatenate([Xa,Xb]),np.concatenate([ya,yb]); c=int(0.85*len(Xp))
        pm=np.random.RandomState(sd).permutation(len(Xp)); Xp,yp=Xp[pm],yp[pm]
        m_pool  = fit_model(Xp[:c],yp[:c],Xp[c:],yp[c:],sd)
        p=rollout(m_pool,S_t,o,H);                      rows['B3'].append(rmse(tgt_true,p)); preds.setdefault('B3',p)
        if verbose: print('   seed %d done'%sd, {k:round(v[-1],5) for k,v in rows.items()})
        K.clear_session(); gc.collect()
    return rows, preds, dict(floor=TGT['FLOOR'], curve_only=curve_only, scale_cal=scale_cal,
                             tgt_true=tgt_true, origin=o, H=H)

print('=== FORWARD: train B0006 -> forecast B0005 ===')
t0=time.time(); FW_rows, FW_preds, FW_meta = run_direction('B0006','B0005')
print('forward done in %.1f min'%((time.time()-t0)/60))

=== FORWARD: train B0006 -> forecast B0005 ===
   seed 42 done {'A1': 0.04267, 'A2': 0.02003, 'A3': 0.04719, 'B1': 0.00727, 'B2': 0.00902, 'B3': 0.01922}
   seed 43 done {'A1': 0.02302, 'A2': 0.02157, 'A3': 0.02648, 'B1': 0.00873, 'B2': 0.00836, 'B3': 0.04137}
   seed 44 done {'A1': 0.02693, 'A2': 0.02218, 'A3': 0.03084, 'B1': 0.00696, 'B2': 0.00688, 'B3': 0.01557}
   seed 45 done {'A1': 0.02241, 'A2': 0.01875, 'A3': 0.02607, 'B1': 0.02076, 'B2': 0.00701, 'B3': 0.0348}
   seed 46 done {'A1': 0.03364, 'A2': 0.01739, 'A3': 0.03775, 'B1': 0.01303, 'B2': 0.0241, 'B3': 0.02719}
forward done in 5.2 min


In [5]:
LABEL={'A1':'Tier 0  zero-shot, plain',
       'A2':'Tier 1  zero-shot + fade re-anchored',
       'A3':'Tier 1  zero-shot + delta calibration',
       'B1':'Tier 2  within-cell only (INCUMBENT)',
       'B2':'Tier 2  pre-train then fine-tune',
       'B3':'Tier 2  pooled training'}

def report(rows, meta, src, tgt):
    floor=meta['floor']; base=np.array(rows['B1'])
    print('\n'+'='*94)
    print('train %s  ->  forecast %s   (origin index %d, %d cycles)'%(src,tgt,meta['origin'],meta['H']))
    print('flat-line baseline %.5f | fade-curve extrapolation alone %.5f | delta calibration x%.3f'
          %(floor,meta['curve_only'],meta['scale_cal']))
    print('='*94)
    print('%-40s %9s %9s %8s %9s %9s'%('condition','RMSE','sd','skill','vs floor','vs B1 p'))
    out=[]
    for k in ['A1','A2','A3','B1','B2','B3']:
        v=np.array(rows[k]); sk=1-v.mean()/floor
        beats='YES' if v.mean()<floor else 'no'
        p=ttest_rel(v,base).pvalue if k!='B1' else np.nan
        print('%-40s %9.5f %9.5f %+8.3f %9s %9s'
              %(LABEL[k],v.mean(),v.std(ddof=1),sk,beats,'--' if k=='B1' else '%.4f'%p))
        out.append(dict(direction='%s->%s'%(src,tgt),cond=k,label=LABEL[k],mean=v.mean(),
                        sd=v.std(ddof=1),skill=sk,beats_floor=beats,p_vs_B1=p))
    print('-'*94)
    a1=np.array(rows['A1'])
    t,p=ttest_rel(a1,np.full_like(a1,floor))
    print('KEY TEST — does a model that never saw %s beat persistence on it?'%tgt)
    print('   zero-shot %.5f vs flat line %.5f   diff %+.5f   p %.4f   -> %s'
          %(a1.mean(),floor,a1.mean()-floor,p,
            'YES, it learned transferable dynamics' if (a1.mean()<floor and p<0.05)
            else 'not at p<0.05 on %d seeds'%len(a1)))
    return out

R=report(FW_rows,FW_meta,'B0006','B0005')


train B0006  ->  forecast B0005   (origin index 133, 35 cycles)
flat-line baseline 0.02927 | fade-curve extrapolation alone 0.01439 | delta calibration x1.081
condition                                     RMSE        sd    skill  vs floor   vs B1 p
Tier 0  zero-shot, plain                   0.02974   0.00850   -0.016        no    0.0279
Tier 1  zero-shot + fade re-anchored       0.01998   0.00197   +0.317       YES    0.0559
Tier 1  zero-shot + delta calibration      0.03367   0.00890   -0.150        no    0.0164
Tier 2  within-cell only (INCUMBENT)       0.01135   0.00579   +0.612       YES        --
Tier 2  pre-train then fine-tune           0.01107   0.00734   +0.622       YES    0.9478
Tier 2  pooled training                    0.02763   0.01068   +0.056       YES    0.0180
----------------------------------------------------------------------------------------------
KEY TEST — does a model that never saw B0005 beat persistence on it?
   zero-shot 0.02974 vs flat line 0.02927   di

## Reverse direction — the same question with the cells swapped

In [6]:
print('=== REVERSE: train B0005 -> forecast B0006 ===')
t0=time.time(); RV_rows, RV_preds, RV_meta = run_direction('B0005','B0006')
print('reverse done in %.1f min'%((time.time()-t0)/60))
R+=report(RV_rows,RV_meta,'B0005','B0006')
T=pd.DataFrame(R); T.to_csv('battery_transfer_results.csv',index=False)

=== REVERSE: train B0005 -> forecast B0006 ===
   seed 42 done {'A1': 0.04829, 'A2': 0.04784, 'A3': 0.04817, 'B1': 0.01652, 'B2': 0.023, 'B3': 0.00802}
   seed 43 done {'A1': 0.02954, 'A2': 0.04338, 'A3': 0.03076, 'B1': 0.00751, 'B2': 0.01866, 'B3': 0.04511}
   seed 44 done {'A1': 0.02815, 'A2': 0.04621, 'A3': 0.02958, 'B1': 0.01994, 'B2': 0.03095, 'B3': 0.0182}
   seed 45 done {'A1': 0.03029, 'A2': 0.04113, 'A3': 0.03143, 'B1': 0.01994, 'B2': 0.01984, 'B3': 0.04277}
   seed 46 done {'A1': 0.03806, 'A2': 0.04094, 'A3': 0.03865, 'B1': 0.00849, 'B2': 0.00852, 'B3': 0.02797}
reverse done in 4.8 min

train B0005  ->  forecast B0006   (origin index 133, 35 cycles)
flat-line baseline 0.04608 | fade-curve extrapolation alone 0.01246 | delta calibration x0.925
condition                                     RMSE        sd    skill  vs floor   vs B1 p
Tier 0  zero-shot, plain                   0.03487   0.00844   +0.243       YES    0.0134
Tier 1  zero-shot + fade re-anchored       0.04390   0.00

## Figures and verdict

In [7]:
fig,axes=plt.subplots(1,2,figsize=(14,4.6))
for ax,(preds,meta,src,tgt) in zip(axes,[(FW_preds,FW_meta,'B0006','B0005'),
                                         (RV_preds,RV_meta,'B0005','B0006')]):
    o,H=meta['origin'],meta['H']; k=np.arange(o,o+H)
    ax.plot(np.arange(len(CELLS[tgt]['S'])),CELLS[tgt]['S'],color='0.35',lw=1.4,label='%s actual'%tgt)
    ax.axvline(o,color='0.7',ls=':',lw=1)
    ax.plot(k,np.full(H,CELLS[tgt]['S'][o-1]),color='0.6',ls='--',lw=1.2,label='flat line')
    for c,st in [('A1','-'),('A2','-'),('B1','-')]:
        if c in preds: ax.plot(k,preds[c],st,lw=1.6,label=LABEL[c].split('  ',1)[1])
    ax.set_title('train %s -> forecast %s'%(src,tgt)); ax.set_xlabel('cycle'); ax.set_ylabel('SOH')
    ax.legend(fontsize=7,loc='lower left'); ax.grid(alpha=.25)
plt.tight_layout(); plt.savefig('battery_transfer_forecasts.png',dpi=150)
print('saved battery_transfer_forecasts.png'); plt.close()

print('\n'+'='*94)
print('VERDICT')
print('='*94)
for d in T.direction.unique():
    sub=T[T.direction==d]; a1=sub[sub.cond=='A1'].iloc[0]; b1=sub[sub.cond=='B1'].iloc[0]
    bestT=sub[sub.cond.isin(['A2','A3'])].nsmallest(1,'mean').iloc[0]
    fl=FW_meta['floor'] if d.endswith('B0005') else RV_meta['floor']
    print('\n%s'%d)
    print('   zero-shot, no target data at all : %.5f   (flat line %.5f)  %s'
          %(a1['mean'],fl,'BEATS persistence' if a1['mean']<fl else 'does NOT beat persistence'))
    print('   best calibrated transfer         : %.5f   via %s'%(bestT['mean'],bestT.cond))
    print('   within-cell incumbent            : %.5f'%b1['mean'])
    print('   cost of never seeing the target  : %.5f  (%.1fx)'%(a1['mean']-b1['mean'],a1['mean']/b1['mean']))
json.dump(dict(forward={k:list(map(float,v)) for k,v in FW_rows.items()},
               reverse={k:list(map(float,v)) for k,v in RV_rows.items()},
               floors={n:CELLS[n]['FLOOR'] for n in CELLS},
               fades={n:float(CELLS[n]['fade']) for n in CELLS}),
          open('battery_transfer_summary.json','w'),indent=2)
print('\nsaved battery_transfer_results.csv and battery_transfer_summary.json')
print('cwd:',os.getcwd())

saved battery_transfer_forecasts.png

VERDICT

B0006->B0005
   zero-shot, no target data at all : 0.02974   (flat line 0.02927)  does NOT beat persistence
   best calibrated transfer         : 0.01998   via A2
   within-cell incumbent            : 0.01135
   cost of never seeing the target  : 0.01839  (2.6x)

B0005->B0006
   zero-shot, no target data at all : 0.03487   (flat line 0.04608)  BEATS persistence
   best calibrated transfer         : 0.03572   via A3
   within-cell incumbent            : 0.01448
   cost of never seeing the target  : 0.02039  (2.4x)

saved battery_transfer_results.csv and battery_transfer_summary.json
cwd: c:\Users\Kamal\Documents\AUC\thesiswork\Nasa_battery_dataset
